In [1]:
import torch
from transformers import pipeline

print(f"Transformers installed: True")
print(f"PyTorch: {torch.__version__}")
print(f"Day 8 - HuggingFace")

Transformers installed: True
PyTorch: 2.12.0+cu130
Day 8 - HuggingFace


In [ ]:
print(f"=== HuggingFace Pipeline API ===\n")
print("Download models - first run takes 1-2 minutes....\n")

# Sentiment Analysis
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english", 
    device=0 if torch.cuda.is_available() else -1
)

texts = [
    "This RAG pipeline is incredibly powerful",
    "The retrieval quality is terrible and unreliable",
    "vector databases are an interesting technology",
    "I hate when models halluciante wrong answers"
]

print("=== Sentiment Analysis===")
results = sentiment(texts)
for text, result in zip(texts, results):
    print(f"[{result['label']} {result['score']:.4f}] {text}")

=== HuggingFace Pipeline API ===

Download models - first run takes 1-2 minutes....



Device set to use cuda:0


=== Sentiment Analysis===
[POSITIVE 0.9997] This RAG pipeline is incredibly powerful
[NEGATIVE 0.9997] The retrieval quality is terrible and unreliable
[POSITIVE 0.9995] vector databases are an interesting technology
[NEGATIVE 0.9987] I hate when models halluciante wrong answers


In [4]:
print("=== Named Entity Recognition ===")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

ner_text = "Anthropic built Claude and is based in San Francisco. OpenAI created GPT-4."
entities = ner(ner_text)
print(f"Text: {ner_text}\n")
for entity in entities:
    print(f"  [{entity['entity_group']}] '{entity['word']}' (score: {entity['score']:.4f})")

print("\n=== Text Summarization ===")
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=0 if torch.cuda.is_available() else -1
)

long_text = """
Retrieval Augmented Generation (RAG) is a technique that combines 
the power of large language models with external knowledge retrieval. 
Instead of relying solely on the model's training data, RAG systems 
first retrieve relevant documents from a knowledge base, then use 
those documents as context when generating responses. This approach 
significantly reduces hallucinations and allows models to access 
up-to-date information beyond their training cutoff. Enterprise RAG 
systems add additional layers including hybrid search combining 
keyword and semantic retrieval, re-ranking for improved precision, 
and evaluation frameworks to measure response quality.
"""

summary = summarizer(long_text, max_length=60, min_length=20, do_sample=False)
print(f"Original length: {len(long_text.split())} words")
print(f"Summary: {summary[0]['summary_text']}")

print("\n=== Question Answering ===")
qa = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=0 if torch.cuda.is_available() else -1
)

context = """
Enterprise RAG pipelines use hybrid search combining BM25 keyword 
search with vector-based semantic search. Results from both methods 
are merged using Reciprocal Rank Fusion. A cross-encoder re-ranker 
then reorders the top candidates for maximum precision. The system 
uses RAGAs framework to evaluate faithfulness and answer relevancy.
"""

questions = [
    "What search methods does Enterprise RAG use?",
    "How are results from different search methods combined?",
    "What framework is used for evaluation?"
]

for question in questions:
    answer = qa(question=question, context=context)
    print(f"  Q: {question}")
    print(f"  A: {answer['answer']} (score: {answer['score']:.4f})\n")

=== Named Entity Recognition ===


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Text: Anthropic built Claude and is based in San Francisco. OpenAI created GPT-4.

  [ORG] 'Anthropic' (score: 0.9860)
  [MISC] 'Claude' (score: 0.9717)
  [LOC] 'San Francisco' (score: 0.9984)
  [ORG] 'OpenAI' (score: 0.9976)
  [MISC] 'GPT' (score: 0.6788)
  [MISC] '4' (score: 0.8391)

=== Text Summarization ===


Device set to use cuda:0


Original length: 88 words
Summary:  Retrieval Augmented Generation (RAG) combines the power of large language models with external knowledge retrieval . This approach reduces hallucinations and allows models to access information beyond their training cutoff .

=== Question Answering ===


Device set to use cuda:0


  Q: What search methods does Enterprise RAG use?
  A: hybrid search combining BM25 keyword 
search with vector-based semantic search (score: 0.3910)

  Q: How are results from different search methods combined?
  A: Reciprocal Rank Fusion (score: 0.7194)

  Q: What framework is used for evaluation?
  A: RAGAs (score: 0.7116)



In [4]:
from transformers import AutoTokenizer

print("=== Tokenization Deep Dive ===\n")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

#Basic tokenization
text = "RAG combines retrieval and generation for better answers"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)
decoded = tokenizer.decode(token_ids)

print(f"Original text: {text}")
print(f"\nTokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"decoded back:{decoded}")
print(f"\nVocab size: {tokenizer.vocab_size}")


# Special tokens
print(f"\nSpecial tokens:")
print(f"  [CLS] ID: {tokenizer.cls_token_id}")
print(f"  [SEP] ID: {tokenizer.sep_token_id}")
print(f"  [PAD] ID: {tokenizer.pad_token_id}")

# Batch tokenization - what you'll do in RAG pipeline

print("\n=== Batch Tokenization ===")
chunks = [
    "RAG combines retrieval and generation",
    "Vector databases store embeddings",
    "Fine tuning adapts pretrained models to specific domains"
]

# Padding and truncation - critical for batching
batch = tokenizer(
    chunks,
    padding = True,         # pad shorter sequence
    truncation = True,      # truncate longer sequences
    max_length=20, 
    return_tensors="pt"     # return pytorch tensors
)

print(f"Input Ids shape: {batch['input_ids'].shape}")
print(f"Attention mask shape: {batch['attention_mask'].shape}")
print(f"\nInput IDs:\n{batch['attention_mask']}")
print("\n1 = real token, 0 = padding token")
print("Model ignores positons where attention_mask = 0")

=== Tokenization Deep Dive ===

Original text: RAG combines retrieval and generation for better answers

Tokens: ['rag', 'combines', 'retrieval', 'and', 'generation', 'for', 'better', 'answers']
Token IDs: [101, 17768, 13585, 26384, 1998, 4245, 2005, 2488, 6998, 102]
decoded back:[CLS] rag combines retrieval and generation for better answers [SEP]

Vocab size: 30522

Special tokens:
  [CLS] ID: 101
  [SEP] ID: 102
  [PAD] ID: 0

=== Batch Tokenization ===
Input Ids shape: torch.Size([3, 13])
Attention mask shape: torch.Size([3, 13])

Input IDs:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

1 = real token, 0 = padding token
Model ignores positons where attention_mask = 0


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("=== Sentence Transformers ====\n")


# This model maps sentences to 384-dimensional embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Your document chunks
chunks = [
    "RAG combines retrieval and generation for better answers",
    "vector databases store and retrieve embeddings efficiently",
    "Python tuning adapts pretrained models to specific domains",
    "Python is a popular programming language for data science",
    "Enterprise RAG uses hybrid search for improved retrieval"
]

# Encode all chunks at once
print("Encoding chunks...")
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Each chunk-> {chunk_embeddings.shape[1]}-dimensional vector")

# Query
query = "how does RAG retrieval work?"
query_embedding = model.encode(query)

print(f"Query: '{query}'")
print(f"Query embedding shape: {query_embedding.shape}")

# Compute similarities using yout retrieve_top_k from Day 3
def cosine_similarity_matrix(query, docs):
    query_norm = query / np.sqrt(np.sum(query**2))
    doc_magnitudes = np.sqrt(np.sum(docs**2, axis = 1, keepdims=True))
    docs_norms = docs/doc_magnitudes
    return np.dot(docs_norms, query_norm)

similarities = cosine_similarity_matrix(query_embedding, chunk_embeddings)
ranked_indices = np.argsort(similarities)[::-1]

print(f"\n=== Semantic Search Results ===")
for rank, idx in enumerate(ranked_indices):
    print(f"    Rank {rank+1}  |  Score: {similarities[idx]:.4f} | {chunks[idx]}")

=== Sentence Transformers ====

Encoding chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (5, 384)
Each chunk-> 384-dimensional vector
Query: 'how does RAG retrieval work?'
Query embedding shape: (384,)

=== Semantic Search Results ===
    Rank 1  |  Score: 0.4969 | RAG combines retrieval and generation for better answers
    Rank 2  |  Score: 0.4817 | Enterprise RAG uses hybrid search for improved retrieval
    Rank 3  |  Score: 0.0705 | vector databases store and retrieve embeddings efficiently
    Rank 4  |  Score: -0.0118 | Python tuning adapts pretrained models to specific domains
    Rank 5  |  Score: -0.0476 | Python is a popular programming language for data science
